# Baseline Model — CryptoShield AI

Starter notebook wiring together the data pipeline (`utils/preprocessing.py`), experiment tracking (`utils/experiment_tracking.py`), and model registry (`utils/model_registry.py`).

Swap in different models / features below and re-run — each run gets logged automatically, and `save_model_version` keeps a versioned copy so nothing is overwritten.

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

import scipy.sparse as sp
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

from utils.preprocessing import run_pipeline
from utils.experiment_tracking import start_experiment, log_run
from utils.model_registry import save_model_version, promote_to_best

## 1. Run the shared data pipeline

This cleans the text, builds the engineered indicator features, splits train/test, and fits TF-IDF — the same pipeline the Streamlit app will use at inference time.

In [ ]:
result = run_pipeline()
train_df, test_df, vectorizer = result["train_df"], result["test_df"], result["vectorizer"]

print(f"Train rows: {len(train_df)} | Test rows: {len(test_df)}")
train_df.head()

## 2. Build the feature matrix

Combines TF-IDF text features with the engineered indicator columns (urgency, wallet/contact, structural).

In [ ]:
engineered_cols = [
    "urgency_keyword_count", "guaranteed_return_keyword_count", "countdown_phrase_count",
    "exclamation_count", "urgency_score", "has_wallet_address", "wallet_address_count",
    "has_url", "url_count", "has_email", "has_phone_number", "payment_keyword_count",
    "off_platform_keyword_count", "credential_keyword_count", "message_length",
    "capital_letter_ratio", "has_numeric_content", "digit_count",
]

X_train_tfidf = vectorizer.transform(train_df["clean_text"])
X_test_tfidf = vectorizer.transform(test_df["clean_text"])

X_train = sp.hstack([X_train_tfidf, train_df[engineered_cols].values])
X_test = sp.hstack([X_test_tfidf, test_df[engineered_cols].values])

y_train = train_df["label"]
y_test = test_df["label"]

SCAM_LABEL = "scam"  # update if the dataset uses a different positive-class label

## 3. Train + log a baseline model

This is the experiment tracking step — everything below the `with log_run(...)` line gets recorded automatically.

In [ ]:
start_experiment()

model = LogisticRegression(max_iter=1000, class_weight="balanced")

with log_run(
    run_name="logreg_baseline",
    params={
        "model": "LogisticRegression",
        "class_weight": "balanced",
        "tfidf_max_features": 5000,
        "tfidf_ngram_range": "(1, 2)",
    },
) as run:
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    probs = model.predict_proba(X_test)[:, list(model.classes_).index(SCAM_LABEL)]

    metrics = {
        "accuracy": accuracy_score(y_test, preds),
        "precision": precision_score(y_test, preds, pos_label=SCAM_LABEL),
        "recall": recall_score(y_test, preds, pos_label=SCAM_LABEL),
        "f1": f1_score(y_test, preds, pos_label=SCAM_LABEL),
        "roc_auc": roc_auc_score((y_test == SCAM_LABEL).astype(int), probs),
    }

    run.log_metrics(metrics)
    run.log_model(model)

print(metrics)
print(classification_report(y_test, preds))
print(confusion_matrix(y_test, preds))

Check against the proposal's success thresholds: Recall >= 0.85, Precision >= 0.75, F1 >= 0.80, ROC-AUC >= 0.85.

To view all logged runs side by side, run in a terminal from the project root: `mlflow ui --backend-store-uri file:./mlruns --port 5000`

## 4. Save this version to the model registry

Only run `promote_to_best` once this version has been reviewed and is the one you want the Streamlit app to use.

In [ ]:
saved = save_model_version(
    model,
    metrics=metrics,
    notes="Logistic Regression baseline, TF-IDF (1,2-gram, 5000 features) + engineered indicators",
)

# Once reviewed and happy with it:
# promote_to_best(saved["version"])

## Next steps

- Try other candidate models (Naive Bayes, SVM, Random Forest) with the same `log_run` / `save_model_version` pattern — each becomes a new tracked run and a new registry version.
- Remember to add a row to `models/MODEL_REGISTRY.md` for each version you keep.
- Once a model is promoted to `best_model.joblib`, the Streamlit app's Scam Detector page can load it instead of the current placeholder logic.